# 비디오

## Part 1. 셋업

In [17]:
import os, random, pickle, time, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    resnet18,        ResNet18_Weights,
    convnext_tiny,   ConvNeXt_Tiny_Weights,
)
from facenet_pytorch import MTCNN

import torch.multiprocessing as mp
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass

print('torch:', torch.__version__)
print('CUDA :', torch.cuda.is_available())

torch: 2.11.0+cu128
CUDA : True


In [18]:
SEED        = 42
IMG_SIZE    = 224
BATCH_SIZE  = 64
FEATURE_DIM = 768   # 멀티모달 fusion용 공통 출력 차원

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ROOT_PATH    = r'C:\Users\user\Documents\50.2026\53.DL_실습\multimodal\data'
VIDEO_FOLDER = os.path.join(ROOT_PATH, 'Video', 'Segmented')
LABEL_PATH   = os.path.join(ROOT_PATH, 'mosi_text_metadata.csv')
PICKLE_PATH  = os.path.join(ROOT_PATH, 'video_preprocessed.pkl')

print(f'device: {device}')
print(f'ROOT_PATH: {ROOT_PATH}')

device: cuda
ROOT_PATH: C:\Users\user\Documents\50.2026\53.DL_실습\multimodal\data


## Part 2. 데이터 로드

In [19]:
label_df = pd.read_csv(LABEL_PATH)
label_df = label_df.dropna(subset=['label'])

video_keys_ordered = [f"{r['video_id']}_{r['seg_idx']}" for _, r in label_df.iterrows()]
labels_ordered     = [int(r['label']) for _, r in label_df.iterrows()]
label_dict         = dict(zip(video_keys_ordered, labels_ordered))

print(f'총 라벨 수: {len(video_keys_ordered)}')
print(f"긍정: {sum(v==1 for v in labels_ordered)}  부정: {sum(v==0 for v in labels_ordered)}")

총 라벨 수: 2199
긍정: 1080  부정: 1119


In [20]:
if os.path.exists(PICKLE_PATH):
    print('캐시 로드:', PICKLE_PATH)
    with open(PICKLE_PATH, 'rb') as f:
        images, labels = pickle.load(f)
    print(f'이미지: {images.shape}  라벨: {labels.shape}')
else:
    print('피클 없음 → 아래 전처리 셀 실행 필요')

캐시 로드: C:\Users\user\Documents\50.2026\53.DL_실습\multimodal\data\video_preprocessed.pkl
이미지: (2199, 224, 224, 3)  라벨: (2199,)


In [21]:
# 피클이 없을 때만 실행
if not os.path.exists(PICKLE_PATH):
    mtcnn = MTCNN(image_size=IMG_SIZE, margin=20, keep_all=False, device=device, post_process=False)
    MTCNN_INPUT_SIZE = (640, 480)

    def extract_middle_frame(video_path):
        cap = cv2.VideoCapture(video_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(total // 2, 0))
        ret, frame = cap.read()
        cap.release()
        if not ret: return None
        return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    def build_dataset():
        face_found = face_missed = 0
        frames_buf, keys_buf, labels_buf = [], [], []
        for key, lbl in tqdm(zip(video_keys_ordered, labels_ordered),
                             total=len(video_keys_ordered), desc='프레임 추출'):
            vpath = os.path.join(VIDEO_FOLDER, key + '.mp4')
            if not os.path.exists(vpath): continue
            img = extract_middle_frame(vpath)
            if img is None: continue
            frames_buf.append(img.resize(MTCNN_INPUT_SIZE, Image.BILINEAR))
            keys_buf.append(key); labels_buf.append(lbl)

        images_out, labels_out = [], []
        for i in tqdm(range(0, len(frames_buf), 64), desc='MTCNN 배치'):
            batch = [f.resize(MTCNN_INPUT_SIZE, Image.BILINEAR) for f in frames_buf[i:i+64]]
            try:   faces = mtcnn(batch)
            except: faces = [None] * len(batch)
            for face, orig, lbl in zip(faces, batch, labels_buf[i:i+64]):
                if face is not None:
                    img_np = face.clamp(0,255).permute(1,2,0).byte().cpu().numpy(); face_found += 1
                else:
                    img_np = np.array(orig.resize((IMG_SIZE, IMG_SIZE))); face_missed += 1
                images_out.append(img_np); labels_out.append(lbl)
        print(f'얼굴 발견: {face_found}  누락: {face_missed}')
        return np.array(images_out), np.array(labels_out)

    images, labels = build_dataset()
    with open(PICKLE_PATH, 'wb') as f: pickle.dump((images, labels), f)
    print('피클 저장:', PICKLE_PATH)

## Part 3. DataLoader & 증강 파이프라인

In [22]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class GaussianNoise:
    def __init__(self, std=0.05): self.std = std
    def __call__(self, t): return (t + torch.randn_like(t) * self.std).clamp(0, 1)

# ── 증강 있음 (실험 A) ────────────────────────────────────────────
train_tf_aug = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0), ratio=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    GaussianNoise(std=0.05),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
])

# ── 증강 없음 (실험 B, 기본 전처리만) ────────────────────────────
train_tf_noaug = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
test_tf = val_tf

print('증강 파이프라인 정의 완료')

증강 파이프라인 정의 완료


In [23]:
class ImageDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = torch.tensor(labels, dtype=torch.long)
        self.transform = transform
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img = self.images[idx]
        return (self.transform(img) if self.transform else img), self.labels[idx]

# train 64% / val 16% / test 20%
X_tr, X_test, y_tr, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=SEED, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(
    X_tr, y_tr, test_size=0.2, random_state=SEED, stratify=y_tr)

NW = 0

def make_loaders(train_tf):
    g = torch.Generator(); g.manual_seed(SEED)
    tr = DataLoader(ImageDataset(X_train, y_train, train_tf),
                    batch_size=BATCH_SIZE, shuffle=True, generator=g,
                    num_workers=NW, pin_memory=True)
    vl = DataLoader(ImageDataset(X_val,   y_val,   val_tf),
                    batch_size=BATCH_SIZE, shuffle=False, num_workers=NW, pin_memory=True)
    te = DataLoader(ImageDataset(X_test,  y_test,  test_tf),
                    batch_size=BATCH_SIZE, shuffle=False, num_workers=NW, pin_memory=True)
    return tr, vl, te

# 증강 있음 / 없음 로더 두 쌍
train_loader_aug,   val_loader, test_loader = make_loaders(train_tf_aug)
train_loader_noaug, _,          _           = make_loaders(train_tf_noaug)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')
print(f'긍정/부정 (Train): {(y_train==1).sum()}/{(y_train==0).sum()}')

Train: 1407  Val: 352  Test: 440
긍정/부정 (Train): 691/716


$# Part 4. VideoEncoder — EfficientNet-B0 / ResNet-18 / ConvNeXt-Tiny

```
backbone       native_dim   proj → 768-dim → proj → 256-dim
─────────────────────────────────────────────────────────────
EfficientNet-B0   1280      Linear(1280,768) + BN + ReLU + Drop
ResNet-18          512      Linear( 512,768) + BN + ReLU + Drop
ConvNeXt-Tiny      768      LayerNorm(768) — 프로젝션 불필요
```

In [30]:
class VideoEncoder(nn.Module):
    def __init__(self, backbone='convnext', num_classes=2, dropout=0.4):
        super().__init__()
        self.backbone_name = backbone

        if backbone == 'efficientnet':
            base        = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
            native_dim  = base.classifier[1].in_features   # 1280
            base.classifier = nn.Identity()
            self.backbone   = base
            self.proj_768   = nn.Sequential(
                nn.Linear(native_dim, 768),
                nn.BatchNorm1d(768), nn.ReLU(), nn.Dropout(dropout),
            )

        elif backbone == 'resnet18':
            base        = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
            native_dim  = base.fc.in_features              # 512
            base.fc     = nn.Identity()
            self.backbone   = base
            self.proj_768   = nn.Sequential(
                nn.Linear(native_dim, 768),
                nn.BatchNorm1d(768), nn.ReLU(), nn.Dropout(dropout),
            )

        elif backbone == 'convnext':
            base = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
            base.classifier[-1] = nn.Identity()   # Linear(768,1000) 제거
            self.backbone = base                   # forward: features→avgpool→flatten→LN→Identity = (B,768)
            self.proj_768 = nn.Dropout(dropout)

        else:
            raise ValueError(f'Unknown backbone: {backbone}')

        self.proj_256 = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(dropout),
        )
        self.head = nn.Linear(256, num_classes)

    # forward
    def _encode_768(self, x):
        return self.proj_768(self.backbone(x))   # (B, 768)

    def forward(self, x):
        return self.head(self.proj_256(self._encode_768(x)))

    def get_features(self, x):
        f768 = self._encode_768(x)
        f256 = self.proj_256(f768)
        return f768, f256

    def extract_features(self, loader, device, dim=768):
        assert dim in (768, 256)
        self.eval()
        feats, ys = [], []
        with torch.no_grad():
            for x, y in tqdm(loader, desc=f'extract {dim}-dim'):
                f768, f256 = self.get_features(x.to(device))
                feats.append((f768 if dim == 768 else f256).cpu())
                ys.append(y)
        return torch.cat(feats), torch.cat(ys)


# 파라미터 수 확인
for bk in ['efficientnet', 'resnet18', 'convnext']:
    m = VideoEncoder(backbone=bk)
    total = sum(p.numel() for p in m.parameters())
    print(f'{bk:15s}: {total:>12,} params')
    del m

efficientnet   :    5,190,782 params
resnet18       :   11,769,922 params
convnext       :   28,018,018 params


## Part 5. 학습 유틸리티

In [ ]:
# MixUp
def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_criterion(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)


# run_epoch 
def run_epoch(model, loader, criterion, optimizer, device,
              train=True, use_mixup=False):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    with (torch.enable_grad() if train else torch.no_grad()):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if train:
                if use_mixup: x, ya, yb, lam = mixup_data(x, y)
                optimizer.zero_grad()
            out  = model(x)
            loss = (mixup_criterion(criterion, out, ya, yb, lam)
                    if (train and use_mixup) else criterion(out, y))
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * y.size(0)
            correct    += (out.argmax(1) == y).sum().item()
            total      += y.size(0)
    return total_loss / total, correct / total


# EarlyStopping
class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_val   = -float('inf')
        self.counter    = 0
        self.best_state = None
        self.stopped_ep = 0

    def step(self, val_acc, model):
        if val_acc > self.best_val + self.min_delta:
            self.best_val   = val_acc
            self.counter    = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience


print('유틸리티 정의 완료')

유틸리티 정의 완료


## Part 6. 2단계 학습 함수

In [ ]:
def train_two_stage(backbone, train_loader, val_loader, test_loader, device,
                    use_mixup=True, label_smoothing=0.1,
                    s1_epochs=10, s2_max=50, s2_patience=10, warmup=5):

    model = VideoEncoder(backbone=backbone).to(device)

    # Stage 1
    print(f'\n[{backbone}] Stage 1: 백본 동결 ({s1_epochs} epochs)')
    for p in model.backbone.parameters():
        p.requires_grad = False

    opt1  = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=1e-3, weight_decay=1e-2)
    sch1  = optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=s1_epochs)
    crit1 = nn.CrossEntropyLoss()

    hist_s1 = {'tr': [], 'val': [], 'te': []}
    for ep in range(1, s1_epochs + 1):
        _, tr  = run_epoch(model, train_loader, crit1, opt1, device, train=True)
        _, val = run_epoch(model, val_loader,   crit1, None, device, train=False)
        _, te  = run_epoch(model, test_loader,  crit1, None, device, train=False)
        sch1.step()
        hist_s1['tr'].append(tr); hist_s1['val'].append(val); hist_s1['te'].append(te)
        print(f'  S1 Ep {ep:02d}/{s1_epochs}  train={tr:.4f}  val={val:.4f}  test={te:.4f}')

    # Stage 2 
    print(f'[{backbone}] Stage 2: 전체 미세조정 (최대 {s2_max} epochs + ES)')
    for p in model.backbone.parameters():
        p.requires_grad = True

    bb_ids   = set(id(p) for p in model.backbone.parameters())
    other_ps = [p for p in model.parameters() if id(p) not in bb_ids]
    opt2 = optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': 1e-5},
        {'params': other_ps,                    'lr': 1e-4},
    ], weight_decay=5e-3)

    def lr_lambda(ep):
        if ep < warmup: return (ep + 1) / warmup
        p = (ep - warmup) / (s2_max - warmup)
        return 0.5 * (1 + np.cos(np.pi * p))

    sch2    = optim.lr_scheduler.LambdaLR(opt2, lr_lambda=lr_lambda)
    crit2   = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    stopper = EarlyStopping(patience=s2_patience)

    hist_s2 = {'tr': [], 'val': [], 'te': []}
    for ep in range(1, s2_max + 1):
        _, tr  = run_epoch(model, train_loader, crit2, opt2, device, train=True, use_mixup=use_mixup)
        _, val = run_epoch(model, val_loader,   crit2, None, device, train=False)
        _, te  = run_epoch(model, test_loader,  crit2, None, device, train=False)
        sch2.step()
        hist_s2['tr'].append(tr); hist_s2['val'].append(val); hist_s2['te'].append(te)
        done = stopper.step(val, model)
        mark = '  ← best' if stopper.counter == 0 else f'  (p {stopper.counter}/{s2_patience})'
        print(f'  S2 Ep {ep:02d}/{s2_max}  train={tr:.4f}  val={val:.4f}  test={te:.4f}{mark}')
        if done:
            print(f'  ★ EarlyStopping @ ep {ep}  best_val={stopper.best_val:.4f}')
            stopper.stopped_ep = ep; break

    if stopper.stopped_ep == 0: stopper.stopped_ep = s2_max
    model.load_state_dict({k: v.to(device) for k, v in stopper.best_state.items()})
    print(f'  완료 — best_val={stopper.best_val:.4f}')
    return model, hist_s1, hist_s2, stopper


print('train_two_stage() 정의 완료')

train_two_stage() 정의 완료


---
## 실험 1. 세 모델 비교 (증강 있음)

EfficientNet-B0 / ResNet-18 / ConvNeXt-Tiny 를 동일 조건(2단계 학습 + MixUp)으로 비교

In [27]:
print('=' * 60)
print('실험 1-A: EfficientNet-B0')
print('=' * 60)
eff_model, eff_s1, eff_s2, eff_stop = train_two_stage(
    'efficientnet', train_loader_aug, val_loader, test_loader, device
)

실험 1-A: EfficientNet-B0

[efficientnet] Stage 1: 백본 동결 (10 epochs)
  S1 Ep 01/10  train=0.5160  val=0.5085  test=0.5159
  S1 Ep 02/10  train=0.5522  val=0.4631  test=0.4568
  S1 Ep 03/10  train=0.5821  val=0.4830  test=0.5136
  S1 Ep 04/10  train=0.5650  val=0.4744  test=0.4705
  S1 Ep 05/10  train=0.6020  val=0.4830  test=0.4591
  S1 Ep 06/10  train=0.5935  val=0.4318  test=0.4500
  S1 Ep 07/10  train=0.5970  val=0.5057  test=0.4977
  S1 Ep 08/10  train=0.6162  val=0.4830  test=0.4705
  S1 Ep 09/10  train=0.6254  val=0.4716  test=0.4500
  S1 Ep 10/10  train=0.6119  val=0.4801  test=0.5000
[efficientnet] Stage 2: 전체 미세조정 (최대 50 epochs + ES)
  S2 Ep 01/50  train=0.5316  val=0.4943  test=0.5182  ← best
  S2 Ep 02/50  train=0.5529  val=0.4886  test=0.4909  (p 1/10)
  S2 Ep 03/50  train=0.5558  val=0.4886  test=0.5159  (p 2/10)
  S2 Ep 04/50  train=0.5892  val=0.4631  test=0.4864  (p 3/10)
  S2 Ep 05/50  train=0.5721  val=0.5028  test=0.5182  ← best
  S2 Ep 06/50  train=0.5558  val=0.5170 

In [28]:
print('=' * 60)
print('실험 1-B: ResNet-18')
print('=' * 60)
res_model, res_s1, res_s2, res_stop = train_two_stage(
    'resnet18', train_loader_aug, val_loader, test_loader, device
)

실험 1-B: ResNet-18

[resnet18] Stage 1: 백본 동결 (10 epochs)
  S1 Ep 01/10  train=0.5139  val=0.4915  test=0.5523
  S1 Ep 02/10  train=0.5672  val=0.5000  test=0.5091
  S1 Ep 03/10  train=0.5729  val=0.4886  test=0.5205
  S1 Ep 04/10  train=0.5736  val=0.5341  test=0.5795
  S1 Ep 05/10  train=0.5757  val=0.4886  test=0.4977
  S1 Ep 06/10  train=0.6048  val=0.4972  test=0.5273
  S1 Ep 07/10  train=0.5984  val=0.5170  test=0.5409
  S1 Ep 08/10  train=0.6119  val=0.5114  test=0.5318
  S1 Ep 09/10  train=0.6262  val=0.4972  test=0.5386
  S1 Ep 10/10  train=0.6347  val=0.4972  test=0.5341
[resnet18] Stage 2: 전체 미세조정 (최대 50 epochs + ES)
  S2 Ep 01/50  train=0.5195  val=0.5227  test=0.5205  ← best
  S2 Ep 02/50  train=0.5480  val=0.5057  test=0.5250  (p 1/10)
  S2 Ep 03/50  train=0.5373  val=0.5142  test=0.5091  (p 2/10)
  S2 Ep 04/50  train=0.5729  val=0.5028  test=0.5227  (p 3/10)
  S2 Ep 05/50  train=0.5565  val=0.5114  test=0.5364  (p 4/10)
  S2 Ep 06/50  train=0.5515  val=0.5114  test=0.5114

In [ ]:
print('=' * 60)
print('실험 1-C: ConvNeXt-Tiny')
print('=' * 60)
cvx_model, cvx_s1, cvx_s2, cvx_stop = train_two_stage(
    'convnext', train_loader_aug, val_loader, test_loader, device
)

실험 1-C: ConvNeXt-Tiny

[convnext] Stage 1: 백본 동결 (10 epochs)
  S1 Ep 01/10  train=0.5430  val=0.5227  test=0.5591
  S1 Ep 02/10  train=0.5707  val=0.5625  test=0.5864
  S1 Ep 03/10  train=0.5693  val=0.5227  test=0.5477
  S1 Ep 04/10  train=0.5828  val=0.5682  test=0.5636
  S1 Ep 05/10  train=0.6091  val=0.5369  test=0.5864
  S1 Ep 06/10  train=0.6048  val=0.5455  test=0.6023
  S1 Ep 07/10  train=0.6084  val=0.5767  test=0.5818
  S1 Ep 08/10  train=0.5999  val=0.5625  test=0.5886
  S1 Ep 09/10  train=0.6027  val=0.5625  test=0.5841
  S1 Ep 10/10  train=0.6105  val=0.5653  test=0.6000
[convnext] Stage 2: 전체 미세조정 (최대 50 epochs + ES)
  S2 Ep 01/50  train=0.5473  val=0.5824  test=0.5682  ← best
  S2 Ep 02/50  train=0.5380  val=0.5795  test=0.5727  (p 1/10)
  S2 Ep 03/50  train=0.5736  val=0.5909  test=0.5886  ← best
  S2 Ep 04/50  train=0.5288  val=0.5994  test=0.6045  ← best
  S2 Ep 05/50  train=0.5771  val=0.6080  test=0.5977  ← best
  S2 Ep 06/50  train=0.5657  val=0.6108  test=0.6000  

In [ ]:
label_names = ['Negative', 'Positive']

def evaluate(model, loader, device):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in loader:
            preds.extend(model(x.to(device)).argmax(1).cpu().tolist())
            targets.extend(y.tolist())
    acc = sum(p == t for p, t in zip(preds, targets)) / len(targets)
    return acc, preds, targets

results_exp1 = {}
for name, mdl in [('EfficientNet-B0', eff_model),
                  ('ResNet-18',       res_model),
                  ('ConvNeXt-Tiny',   cvx_model)]:
    acc, preds, targets = evaluate(mdl, test_loader, device)
    results_exp1[name] = acc
    print(f'\n=== {name}  test_acc={acc:.4f} ===')
    print(classification_report(targets, preds, target_names=label_names))

print('\n★ 실험 1 요약')
for k, v in sorted(results_exp1.items(), key=lambda x: -x[1]):
    print(f'  {k:20s}: {v:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
model_info = [
    ('EfficientNet-B0', eff_s1, eff_s2, eff_stop),
    ('ResNet-18',       res_s1, res_s2, res_stop),
    ('ConvNeXt-Tiny',   cvx_s1, cvx_s2, cvx_stop),
]
for col, (name, s1, s2, stop) in enumerate(model_info):
    # Stage 1
    ax = axes[0][col]
    ax.plot(range(1, len(s1['tr'])+1), s1['tr'],  label='Train')
    ax.plot(range(1, len(s1['val'])+1), s1['val'], label='Val',  ls='--')
    ax.plot(range(1, len(s1['te'])+1),  s1['te'],  label='Test', ls=':')
    ax.set_title(f'{name}\nStage1 (Frozen)')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(fontsize=8); ax.grid(True)

    # Stage 2
    ax = axes[1][col]
    ax.plot(range(1, len(s2['tr'])+1),  s2['tr'],  label='Train')
    ax.plot(range(1, len(s2['val'])+1), s2['val'], label='Val',  ls='--')
    ax.plot(range(1, len(s2['te'])+1),  s2['te'],  label='Test', ls=':')
    ax.axvline(stop.stopped_ep, color='red', ls='-.', lw=1.2,
               label=f'ES ep{stop.stopped_ep}')
    ax.set_title(f'{name}\nStage2 (Fine-tune, test={results_exp1[name]:.3f})')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(fontsize=8); ax.grid(True)

plt.suptitle('실험 1: 세 모델 학습 곡선 비교', fontsize=13)
plt.tight_layout()
plt.savefig('exp1_model_comparison.png', dpi=150)
plt.show()

# 막대 그래프
fig, ax = plt.subplots(figsize=(7, 4))
names = list(results_exp1.keys())
accs  = [results_exp1[n] for n in names]
colors = ['#4C72B0' if n != max(results_exp1, key=results_exp1.get) else '#DD8452' for n in names]
bars = ax.bar(names, accs, color=colors)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.4f}', ha='center', va='bottom', fontsize=11)
ax.set_ylim(0, 1); ax.set_ylabel('Test Accuracy')
ax.set_title('실험 1: 모델별 Test Accuracy 비교')
ax.axhline(0.7, color='gray', ls='--', lw=1, label='목표 70%')
ax.legend(); ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('exp1_bar_chart.png', dpi=150)
plt.show()

---
## 실험 2. 증강 효과 검증 (최고 성능 모델)

실험 1에서 가장 높은 test_acc를 기록한 모델로, **증강 있음 vs 없음** 비교

In [ ]:
best_backbone = max(results_exp1, key=results_exp1.get)
backbone_key  = {'EfficientNet-B0': 'efficientnet',
                 'ResNet-18':       'resnet18',
                 'ConvNeXt-Tiny':   'convnext'}[best_backbone]
print(f'★ 선택된 모델: {best_backbone} ({backbone_key})')
print(f'  실험 1 test_acc: {results_exp1[best_backbone]:.4f}')

In [ ]:
print('=' * 60)
print(f'실험 2-A: {best_backbone} — 증강 없음 (No Augmentation)')
print('=' * 60)
noaug_model, noaug_s1, noaug_s2, noaug_stop = train_two_stage(
    backbone_key, train_loader_noaug, val_loader, test_loader, device,
    use_mixup=False, label_smoothing=0.0
)

In [ ]:
# 증강 있음 모델은 이미 학습됨 (실험 1 결과 재사용)
aug_model = {'efficientnet': eff_model,
             'resnet18':     res_model,
             'convnext':     cvx_model}[backbone_key]
aug_s2    = {'efficientnet': eff_s2,
             'resnet18':     res_s2,
             'convnext':     cvx_s2}[backbone_key]
aug_stop  = {'efficientnet': eff_stop,
             'resnet18':     res_stop,
             'convnext':     cvx_stop}[backbone_key]

acc_aug,   preds_aug,   tgt = evaluate(aug_model,   test_loader, device)
acc_noaug, preds_noaug, _   = evaluate(noaug_model, test_loader, device)

print(f'\n=== {best_backbone} — 증강 있음  test_acc={acc_aug:.4f} ===')
print(classification_report(tgt, preds_aug, target_names=label_names))

print(f'\n=== {best_backbone} — 증강 없음  test_acc={acc_noaug:.4f} ===')
print(classification_report(tgt, preds_noaug, target_names=label_names))

print(f'\n★ 증강 효과: {acc_aug - acc_noaug:+.4f} ({"증강 有 우위" if acc_aug > acc_noaug else "증강 無 우위"})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, s2, stop, acc in [
    (axes[0], f'{best_backbone}\n증강 있음 (Aug)',   aug_s2,   aug_stop,   acc_aug),
    (axes[1], f'{best_backbone}\n증강 없음 (NoAug)', noaug_s2, noaug_stop, acc_noaug),
]:
    eps = range(1, len(s2['tr']) + 1)
    ax.plot(eps, s2['tr'],  label='Train')
    ax.plot(eps, s2['val'], label='Val',  ls='--')
    ax.plot(eps, s2['te'],  label='Test', ls=':')
    ax.axvline(stop.stopped_ep, color='red', ls='-.', lw=1.2,
               label=f'ES ep{stop.stopped_ep}')
    ax.set_title(f'{label}  (test={acc:.4f})')
    ax.set_xlabel('Epoch (Stage 2)'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(True)

plt.suptitle(f'실험 2: {best_backbone} 증강 유/무 비교', fontsize=13)
plt.tight_layout()
plt.savefig('exp2_aug_comparison.png', dpi=150)
plt.show()

# 나란히 막대 비교
fig, ax = plt.subplots(figsize=(6, 4))
x = [0, 1]
bars = ax.bar(x, [acc_aug, acc_noaug], color=['#4C72B0', '#C44E52'], width=0.5)
ax.set_xticks(x); ax.set_xticklabels(['증강 있음 (Aug)', '증강 없음 (NoAug)'])
for bar, acc in zip(bars, [acc_aug, acc_noaug]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.4f}', ha='center', fontsize=12)
ax.set_ylim(0, 1); ax.set_ylabel('Test Accuracy')
ax.set_title(f'{best_backbone}: 증강 유/무 Test Accuracy')
ax.axhline(0.7, color='gray', ls='--', lw=1, label='목표 70%')
ax.legend(); ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('exp2_aug_bar.png', dpi=150)
plt.show()

---
## Part 9. 768-dim & 256-dim 피처 추출 (멀티모달 Fusion 준비)

실험 1 최고 성능 모델(증강 있음)로 전체 데이터 피처 저장

In [ ]:
print(f'피처 추출 모델: {best_backbone}')

all_loader = DataLoader(
    ImageDataset(images, labels, test_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NW
)

video_feats_768, video_labels = aug_model.extract_features(all_loader, device, dim=768)
video_feats_256, _            = aug_model.extract_features(all_loader, device, dim=256)

print(f'768-dim: {video_feats_768.shape}')
print(f'256-dim: {video_feats_256.shape}')

# 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, feats, dim in [(axes[0], video_feats_768, 768), (axes[1], video_feats_256, 256)]:
    ax.hist(feats.numpy().flatten(), bins=60, color='steelblue', alpha=0.7)
    ax.set_title(f'{best_backbone} {dim}-dim feature distribution')
    ax.set_xlabel('Value'); ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('video_feature_dist.png', dpi=150)
plt.show()

# 저장
for feats, fname in [(video_feats_768, 'video_features_768.pkl'),
                     (video_feats_256, 'video_features_256.pkl')]:
    path = os.path.join(ROOT_PATH, fname)
    with open(path, 'wb') as f:
        pickle.dump({'features': feats, 'labels': video_labels}, f)
    print(f'저장: {path}')

---
## 최종 요약

### 실험 1: 모델 비교
| 모델 | Test Acc |
|------|----------|
| EfficientNet-B0 | — |
| ResNet-18 | — |
| ConvNeXt-Tiny | — |

## 실험 2: 증강 효과
| 조건 | Test Acc |
|------|----------|
| 증강 있음 | — |
| 증강 없음 | — |

## 저장 파일
- `data/video_features_768.pkl` → `{'features': Tensor(N,768), 'labels': Tensor(N,)}`
- `data/video_features_256.pkl` → `{'features': Tensor(N,256), 'labels': Tensor(N,)}`